# Final evaluation - inference only (no retraining)

Loads the saved v7 artifacts (the run that scored 82) and predicts the 1,948
test images in `final_test_set/`.

**Settings**
- Accelerator: **GPU T4 x2** (CLIP on cuda:0, DINOv2 on cuda:1; falls back to one GPU)
- Internet: **OFF**. Everything loads from the attached artifacts, which is both
  the rule-4.3 requirement and the proof of it. If an import fails, turn Internet
  on, run Cell 0, then turn it back off is not possible mid-session - just leave it
  on, since package installs are not part of generating a location guess.

**Inputs attached**: the competition dataset, plus the v7 notebook output. v6 may
also be attached - Cell 2 explicitly ignores it (see below).

**Why this notebook does not just glob for artifacts:** v6 and v7 both wrote
`calibration.json`, `clip_vit_l14/` and head weights, but with *different*
architectures (v6: CLIP-only, 2048-dim, hidden 1024; v7: CLIP+DINOv2, 4096-dim).
A recursive glob would mix them and load silently-wrong weights. Cell 2 picks one
artifact directory and validates every file against the checkpoint's own shapes.

In [1]:
# =====================================================================
# CELL 0 - ONLY run this if Cell 1 reports a missing package.
# With Internet OFF this cell will fail; that is expected and fine.
# =====================================================================
# !pip install -q transformers shapely
print("skip unless an import failed below", flush=True)

skip unless an import failed below


In [2]:
# =====================================================================
# CELL 1 - Offline mode, imports, helpers
# =====================================================================
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["PYTHONHASHSEED"] = "42"
import sys, gc, json, math, time, glob, random, warnings, traceback
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
from transformers import CLIPVisionModel, AutoModel

T0 = time.time()
def log(m): print(f"[+{(time.time()-T0)/60:5.1f} min] {m}", flush=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

R_EARTH = 6371.0088
IMG, N_VIEWS, BS = 224, 2, 64
N_GPU = torch.cuda.device_count()
DEV0 = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DEV1 = torch.device("cuda:1") if N_GPU >= 2 else DEV0
log(f"torch {torch.__version__} | GPUs {N_GPU} | "
    + ("CLIP->cuda:0, DINOv2->cuda:1" if N_GPU>=2 else "single device"))

def vec_to_latlon(v):
    v = np.asarray(v, float); v = v/(np.linalg.norm(v,axis=-1,keepdims=True)+1e-12)
    return (np.degrees(np.arcsin(np.clip(v[...,2],-1,1))),
            np.degrees(np.arctan2(v[...,1], v[...,0])))
def hav_km(a,b,c,d):
    a,b,c,d = map(lambda x: np.radians(np.asarray(x,float)), (a,b,c,d))
    h = np.sin((c-a)/2)**2 + np.cos(a)*np.cos(c)*np.sin((d-b)/2)**2
    return 2*R_EARTH*np.arcsin(np.sqrt(np.clip(h,0,1)))
def find_all(pat, root="/kaggle/input"):
    return sorted(glob.glob(os.path.join(root,"**",pat), recursive=True))
log("imports OK")

[+  0.0 min] torch 2.10.0+cu128 | GPUs 2 | CLIP->cuda:0, DINOv2->cuda:1
[+  0.0 min] imports OK


In [3]:
# =====================================================================
# CELL 2 - Pick ONE artifact directory (v7) and validate it.
# v6 and v7 both wrote calibration.json with incompatible architectures,
# so this must be chosen deliberately, not globbed.
# =====================================================================
cands = sorted(set(os.path.dirname(p) for p in find_all("calibration.json")))
log(f"artifact dirs found: {cands}")
assert cands, "no calibration.json anywhere - is the v7 notebook output attached?"

def score_dir(d):
    """Prefer a dir that is v7-shaped: has DINOv2 + head_seed weights + v7 in path."""
    s = 0
    if "v7" in d.lower(): s += 100
    if os.path.exists(os.path.join(d,"dinov2_large","config.json")): s += 50
    if glob.glob(os.path.join(d,"head_seed*.pt")): s += 25
    return s

ART = max(cands, key=score_dir)
log(f">>> USING ARTIFACTS FROM: {ART}")
for d in cands:
    if d != ART: log(f"    (ignoring {d})")

HEADS = sorted(glob.glob(os.path.join(ART,"head_seed*.pt"))) \
        or sorted(glob.glob(os.path.join(ART,"head_fold*.pt")))
assert HEADS, f"no head weights in {ART}"

MU  = np.load(os.path.join(ART,"feat_mu.npy"))
SD  = np.load(os.path.join(ART,"feat_sd.npy"))
CF  = np.load(os.path.join(ART,"fine_centroids.npy"))
CELL_COUNTRY = np.load(os.path.join(ART,"cell_country.npy"))
CAL = json.load(open(os.path.join(ART,"calibration.json")))

LAM   = float(CAL.get("lam", 0.0))
ALPHA = float(CAL.get("alpha", 4.2))
FLOOR = float(CAL.get("floor", 15.0))
R_MAX = float(CAL.get("r_max", 3000.0))
log(f"calibration.json -> {CAL}")
log(f"using lambda={LAM} alpha={ALPHA} floor={FLOOR} r_max={R_MAX}")

sd0 = torch.load(HEADS[0], map_location="cpu")
FEAT_DIM  = sd0["trunk.0.weight"].shape[1]
HID       = sd0["trunk.0.weight"].shape[0]
N_FINE    = sd0["fine.weight"].shape[0]
N_COARSE  = sd0["coarse.weight"].shape[0]
N_COUNTRY = sd0["country.weight"].shape[0]
log(f"checkpoint shapes: feat_dim={FEAT_DIM} hid={HID} fine={N_FINE} "
    f"coarse={N_COARSE} country={N_COUNTRY} | {len(HEADS)} heads")

# --- hard consistency checks: catch a v6/v7 mix immediately ---
assert MU.shape[0] == FEAT_DIM, f"feat_mu {MU.shape[0]} != head input {FEAT_DIM} (mixed artifacts!)"
assert SD.shape[0] == FEAT_DIM, f"feat_sd {SD.shape[0]} != head input {FEAT_DIM}"
assert CF.shape[0] == N_FINE,   f"centroids {CF.shape[0]} != head fine {N_FINE}"
assert CELL_COUNTRY.shape[0] == N_FINE, "cell_country length != n_fine"
for i,h in enumerate(HEADS):
    s = torch.load(h, map_location="cpu")
    assert s["trunk.0.weight"].shape == sd0["trunk.0.weight"].shape, f"head {i} shape mismatch"
log("consistency checks PASSED - all artifacts come from the same run")

CLIP_LOCAL = os.path.join(ART,"clip_vit_l14")
DINO_LOCAL = os.path.join(ART,"dinov2_large")
assert os.path.exists(os.path.join(CLIP_LOCAL,"config.json")), f"CLIP weights missing in {ART}"
USE_DINO = (FEAT_DIM == 4096)
if USE_DINO:
    assert os.path.exists(os.path.join(DINO_LOCAL,"config.json")), \
        "feature dim 4096 implies DINOv2 was used, but its weights are not in this artifact dir"
log(f"backbones: {'CLIP + DINOv2' if USE_DINO else 'CLIP only'}")

[+  0.3 min] artifact dirs found: ['/kaggle/input/notebooks/mitraasrinivasan1367/geogs-v6/artifacts', '/kaggle/input/notebooks/mitraasrinivasan1367/geogs-v7/artifacts']
[+  0.3 min] >>> USING ARTIFACTS FROM: /kaggle/input/notebooks/mitraasrinivasan1367/geogs-v7/artifacts
[+  0.3 min]     (ignoring /kaggle/input/notebooks/mitraasrinivasan1367/geogs-v6/artifacts)
[+  0.3 min] calibration.json -> {'blend': 0.0, 'lam': 0.0, 'alpha': 4.200000000000001, 'floor': 15.0, 'score': 0.37716166448491106, 'w_provided': 8.0, 'use_dino': True}
[+  0.3 min] using lambda=0.0 alpha=4.200000000000001 floor=15.0 r_max=3000.0
[+  0.3 min] checkpoint shapes: feat_dim=4096 hid=1024 fine=2000 coarse=150 country=298 | 5 heads
[+  0.3 min] consistency checks PASSED - all artifacts come from the same run
[+  0.3 min] backbones: CLIP + DINOv2


In [4]:
# =====================================================================
# CELL 3 - Locate the NEW submission file and the 1,948 test images.
# Note: the file is literally named "sample_submission final.csv" (with a
# space), and training_dataset/ is also mounted - so test images are taken
# from final_test_set/ only, never from the training folder.
# =====================================================================
ss_all = find_all("sample*submission*.csv")
log(f"sample_submission candidates: {ss_all}")
assert ss_all, "no sample submission found"
pref = [p for p in ss_all if "final" in os.path.basename(p).lower()]
SAMPLE_SUB = (pref or ss_all)[0]
if len(ss_all) > 1: log(f">>> chose {SAMPLE_SUB}")
sub_template = pd.read_csv(SAMPLE_SUB)

def pick_col(cols,*k):
    for c in cols:
        lc=c.lower().replace("_","").replace(" ","")
        if all(x in lc for x in k): return c
SUB_ID  = pick_col(sub_template.columns,"id") or sub_template.columns[0]
SUB_LAT = pick_col(sub_template.columns,"lat")
SUB_LON = pick_col(sub_template.columns,"lon") or pick_col(sub_template.columns,"lng")
SUB_RAD = pick_col(sub_template.columns,"rad")
log(f"columns {list(sub_template.columns)} -> {SUB_ID}/{SUB_LAT}/{SUB_LON}/{SUB_RAD}")
log(f"rows to predict: {len(sub_template)}")

# ---- images: restrict to the test folder first ----
test_roots = [d for d in glob.glob("/kaggle/input/**/final_test_set", recursive=True)]
log(f"test roots: {test_roots}")
test_imgs = []
for d in test_roots:
    for ext in ("*.jpg","*.jpeg","*.png","*.JPG","*.JPEG","*.PNG"):
        test_imgs += glob.glob(os.path.join(d,"**",ext), recursive=True)
test_imgs = sorted(set(test_imgs))
log(f"images inside final_test_set: {len(test_imgs)}")

by_stem = {}
for p in test_imgs:
    by_stem.setdefault(os.path.splitext(os.path.basename(p))[0], p)

ids = sub_template[SUB_ID].astype(str).tolist()
TEST_PATHS = [by_stem.get(os.path.splitext(str(t))[0]) for t in ids]
n_missing = sum(p is None for p in TEST_PATHS)

if n_missing:
    log(f"{n_missing} not found under final_test_set - widening search (excluding training_dataset)")
    wide = []
    for ext in ("*.jpg","*.jpeg","*.png"):
        wide += [p for p in find_all(ext) if "training_dataset" not in p and "noised_dataset" not in p]
    for p in sorted(set(wide)):
        by_stem.setdefault(os.path.splitext(os.path.basename(p))[0], p)
    TEST_PATHS = [by_stem.get(os.path.splitext(str(t))[0]) for t in ids]
    n_missing = sum(p is None for p in TEST_PATHS)

log(f"test images matched: {len(TEST_PATHS)-n_missing}/{len(TEST_PATHS)}")
assert n_missing == 0, f"{n_missing} test images could not be located"

[+  0.5 min] sample_submission candidates: ['/kaggle/input/datasets/mitraasrinivasan/geo-guessr-final-hackathon-evaluation/geo-guessr-final-hackathon-evaluation/sample_submission final.csv']
[+  0.5 min] columns ['image_id', 'pred_lat', 'pred_long', 'pred_radius_km'] -> image_id/pred_lat/pred_long/pred_radius_km
[+  0.5 min] rows to predict: 2448
[+  0.6 min] test roots: ['/kaggle/input/datasets/mitraasrinivasan/geo-guessr-final-hackathon-evaluation/geo-guessr-final-hackathon-evaluation/final_test_set']
[+  0.6 min] images inside final_test_set: 2448
[+  0.6 min] test images matched: 2448/2448


In [5]:
# =====================================================================
# CELL 4 - Encode the test images. View construction is byte-identical to
# training: full-frame squash + centre square crop, NO horizontal flip
# (mirroring would destroy driving-side, a strong geolocation cue).
# =====================================================================
CLIP_MEAN,CLIP_STD = [0.48145466,0.4578275,0.40821073],[0.26862954,0.26130258,0.27577711]
DINO_MEAN,DINO_STD = [0.485,0.456,0.406],[0.229,0.224,0.225]

def load_backbone(kind, dev):
    if kind=="clip":
        m = CLIPVisionModel.from_pretrained(CLIP_LOCAL).to(dev).half().eval(); mean,std=CLIP_MEAN,CLIP_STD
    else:
        m = AutoModel.from_pretrained(DINO_LOCAL).to(dev).half().eval(); mean,std=DINO_MEAN,DINO_STD
    for p in m.parameters(): p.requires_grad=False
    return dict(model=m, kind=kind, dim=m.config.hidden_size, dev=dev,
                mean=torch.tensor(mean,device=dev).view(1,3,1,1).half(),
                std =torch.tensor(std, device=dev).view(1,3,1,1).half())

class ViewDS(Dataset):
    def __init__(s, paths): s.p=[x if x else "" for x in paths]
    def __len__(s): return len(s.p)
    def __getitem__(s,i):
        ok=1
        try: im = Image.open(s.p[i]).convert("RGB")
        except Exception: im = Image.new("RGB",(IMG,IMG),(128,128,128)); ok=0
        w,h=im.size; q=min(w,h); l,t=(w-q)//2,(h-q)//2
        a=im.resize((IMG,IMG), Image.BICUBIC)
        b=im.crop((l,t,l+q,t+q)).resize((IMG,IMG), Image.BICUBIC)
        return (torch.stack([torch.from_numpy(np.asarray(a,dtype=np.uint8)).permute(2,0,1),
                             torch.from_numpy(np.asarray(b,dtype=np.uint8)).permute(2,0,1)]), ok)

def _fwd(bk, x_cpu, B):
    x = x_cpu.to(bk["dev"], non_blocking=True).reshape(B*N_VIEWS,3,IMG,IMG).half().div_(255.)
    o = bk["model"](pixel_values=(x-bk["mean"])/bk["std"])
    f = o.pooler_output if bk["kind"]=="clip" else o.last_hidden_state[:,0]
    return f.reshape(B, N_VIEWS*bk["dim"])

@torch.no_grad()
def encode(paths):
    bk_c = load_backbone("clip", DEV0)
    bk_d = load_backbone("dino", DEV1) if USE_DINO else None
    dl = DataLoader(ViewDS(paths), batch_size=BS, shuffle=False, num_workers=4, pin_memory=True)
    A,Bx,bad,seen = [],[],0,0
    for x,ok in dl:
        n = x.shape[0]
        fc = _fwd(bk_c,x,n)                       # queued on cuda:0
        if USE_DINO: fd = _fwd(bk_d,x,n)          # queued on cuda:1 - overlaps
        A.append(fc.float().cpu().numpy())
        if USE_DINO: Bx.append(fd.float().cpu().numpy())
        bad += int((ok==0).sum()); seen += n
        if seen % (BS*10) < BS: log(f"  encoded {seen}/{len(paths)}")
    bk_c["model"].cpu()
    if USE_DINO: bk_d["model"].cpu()
    gc.collect(); torch.cuda.empty_cache()
    Z = np.concatenate([np.concatenate(A), np.concatenate(Bx)],axis=1) if USE_DINO \
        else np.concatenate(A)
    return Z, bad

log(f"encoding {len(TEST_PATHS)} test images ...")
Z, n_bad = encode(TEST_PATHS)
log(f"features {Z.shape} | unreadable images: {n_bad}")
assert Z.shape[1] == FEAT_DIM, f"feature dim {Z.shape[1]} != expected {FEAT_DIM}"
Xt = torch.tensor((Z-MU)/SD, dtype=torch.float32, device=DEV0)

[+  0.6 min] encoding 2448 test images ...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

[+  1.2 min]   encoded 640/2448
[+  1.4 min]   encoded 1280/2448
[+  1.6 min]   encoded 1920/2448
[+  1.8 min] features (2448, 4096) | unreadable images: 0


In [6]:
# =====================================================================
# CELL 5 - Rebuild the heads, run the ensemble, decode
# =====================================================================
CFT = torch.tensor(CF, dtype=torch.float32, device=DEV0)
CCT = torch.tensor(np.where(CELL_COUNTRY>=0, CELL_COUNTRY, 0), device=DEV0)
CC_VALID = torch.tensor((CELL_COUNTRY>=0).astype(np.float32), device=DEV0)

class GeoHead(nn.Module):
    def __init__(s,d,nf,nc,nk,hid):
        super().__init__()
        s.trunk=nn.Sequential(nn.Linear(d,hid),nn.LayerNorm(hid),nn.GELU(),nn.Dropout(0.30),
                              nn.Linear(hid,hid),nn.LayerNorm(hid),nn.GELU(),nn.Dropout(0.15))
        s.fine=nn.Linear(hid,nf); s.coarse=nn.Linear(hid,nc); s.country=nn.Linear(hid,nk)
        s.delta=nn.Linear(hid,3); s.unc=nn.Linear(hid,1)
    def forward(s,x):
        h=s.trunk(x); return s.fine(h),s.coarse(h),s.country(h),s.delta(h),s.unc(h).squeeze(-1)

def decode_point(fl, delta, topk=8, max_spread_km=600.0):
    """Top-K weighted centroid restricted to cells near the top-1 cell - averaging
    cells on opposite continents lands the point in an ocean."""
    p=torch.softmax(fl.float(),1); w,idx=torch.topk(p,topk,1)
    anchor=CFT[idx[:,0]]; cand=CFT[idx]
    cos=(cand*anchor.unsqueeze(1)).sum(-1).clamp(-1+1e-9,1-1e-9)
    w=w*((R_EARTH*torch.acos(cos))<=max_spread_km).float()
    w=w/(w.sum(1,keepdim=True)+1e-9)
    base=(cand*w.unsqueeze(-1)).sum(1); base=base/(base.norm(dim=1,keepdim=True)+1e-9)
    v=base+0.08*torch.tanh(delta); return v/(v.norm(dim=1,keepdim=True)+1e-9)

def country_adjust(fl, cl, lam):
    if lam<=0: return fl
    lp=torch.log_softmax(fl.float(),1); pc=torch.softmax(cl.float(),1)
    return lp + lam*torch.log(pc[:,CCT]*CC_VALID.unsqueeze(0) + 1e-6)

Tf=torch.zeros(len(Xt),N_FINE,device=DEV0); Tc=torch.zeros(len(Xt),N_COUNTRY,device=DEV0)
Td=torch.zeros(len(Xt),3,device=DEV0);      Tu=torch.zeros(len(Xt),device=DEV0)
with torch.no_grad():
    for hp in HEADS:
        m=GeoHead(FEAT_DIM,N_FINE,N_COARSE,N_COUNTRY,HID).to(DEV0)
        m.load_state_dict(torch.load(hp,map_location=DEV0)); m.eval()
        for i in range(0,len(Xt),4096):
            fl,cl,ctl,dl_,ul = m(Xt[i:i+4096])
            Tf[i:i+4096]+=torch.softmax(fl.float(),1); Tc[i:i+4096]+=torch.softmax(ctl.float(),1)
            Td[i:i+4096]+=dl_.float();                 Tu[i:i+4096]+=ul.float()
        del m; log(f"  ensembled {os.path.basename(hp)}")
K=len(HEADS); Tf/=K; Tc/=K; Td/=K; Tu/=K

v = decode_point(country_adjust(torch.log(Tf+1e-12), torch.log(Tc+1e-12), LAM), Td)
p_lat,p_lon = vec_to_latlon(v.cpu().numpy())
p_rad = np.clip(ALPHA*np.maximum(np.expm1(Tu.cpu().numpy()),1.0), FLOOR, R_MAX)
log(f"decoded | radius median {np.median(p_rad):.0f} km "
    f"(min {p_rad.min():.0f}, max {p_rad.max():.0f})")

[+  1.8 min]   ensembled head_seed0.pt
[+  1.8 min]   ensembled head_seed1.pt
[+  1.8 min]   ensembled head_seed2.pt
[+  1.8 min]   ensembled head_seed3.pt
[+  1.8 min]   ensembled head_seed4.pt
[+  1.9 min] decoded | radius median 583 km (min 241, max 2014)


In [7]:
# =====================================================================
# CELL 6 - Ocean rescue + write submission
# =====================================================================
try:
    import shapely
    from shapely.geometry import shape, Point
    from shapely.strtree import STRtree
    from shapely.ops import nearest_points
    gj = find_all("*.geojson")
    log(f"geojson files: {gj}")
    geoms=[]
    if gj:
        for ft_ in json.load(open(gj[0],encoding="utf-8"))["features"]:
            try:
                g=shape(ft_["geometry"]); geoms.append(g if g.is_valid else g.buffer(0))
            except Exception: pass
    if geoms:
        tree=STRtree(geoms)
        pts=shapely.points(p_lon.astype(float), p_lat.astype(float))
        inside=np.zeros(len(pts),bool)
        pr=tree.query(pts,predicate="intersects"); inside[pr[0]]=True
        ocean=np.where(~inside)[0]
        log(f"points outside every country: {len(ocean)}/{len(pts)}")
        for j in ocean:
            try:
                pt=Point(float(p_lon[j]),float(p_lat[j]))
                gi=tree.nearest(pt); gi=int(gi if np.isscalar(gi) else np.asarray(gi).ravel()[0])
                q,_=nearest_points(geoms[gi],pt); p_lat[j],p_lon[j]=q.y,q.x
            except Exception: pass
        log("snapped to nearest land")
    else:
        log("no usable geojson - skipping ocean snap")
except Exception as e:
    log(f"ocean snap skipped ({e})")

p_lat=np.clip(p_lat,-90,90); p_lon=((p_lon+180)%360)-180
bad=~np.isfinite(p_lat)|~np.isfinite(p_lon)|~np.isfinite(p_rad)
if bad.any():
    log(f"WARNING {int(bad.sum())} non-finite -> safe defaults")
    p_lat[bad],p_lon[bad],p_rad[bad]=0.0,0.0,2000.0

sub = sub_template.copy()
sub[SUB_LAT],sub[SUB_LON],sub[SUB_RAD] = p_lat,p_lon,p_rad
sub = sub[list(sub_template.columns)]
OUT="/kaggle/working/submission.csv"; sub.to_csv(OUT,index=False)
log(f"WROTE {OUT}")
print(sub.head(8).to_string())

[+  2.0 min] geojson files: ['/kaggle/input/datasets/mitraasrinivasan/geo-guessr-final-hackathon-evaluation/country_boundaries.geojson', '/kaggle/input/datasets/mitraasrinivasan/geo-guessr-final-hackathon-evaluation/geo-guessr-final-hackathon-evaluation/country_boundaries.geojson']
[+  2.0 min] points outside every country: 447/2448
[+  2.0 min] snapped to nearest land
[+  2.0 min] WROTE /kaggle/working/submission.csv
               image_id   pred_lat   pred_long  pred_radius_km
0  a88766ed092cd328.jpg  56.529447   23.014989      521.068481
1  1d1b147ac111029f.jpg   1.298043  103.751113      282.759338
2  8703a028b9392b48.jpg  56.986762   23.926862      626.263733
3  4366f03b294ff3d7.jpg  17.682766  -64.895172      726.152954
4  251dd7fa406aeaa1.jpg  58.936184   22.044688      581.334595
5  15c3c59ced2f82fe.jpg  61.219081   15.033210      623.453674
6  f4f50202fe33f2b8.jpg  44.529373   20.477668      466.880920
7  ea05b40b6df1e33e.jpg  62.201484   22.821264      695.964539


In [8]:
# =====================================================================
# CELL 7 - PRE-SUBMISSION CHECKS. One upload only. Read every line.
# =====================================================================
ok_all = True
def check(name, cond, detail=""):
    global ok_all
    ok_all &= bool(cond)
    print(f"  [{'PASS' if cond else 'FAIL'}] {name} {detail}", flush=True)

print("="*72)
check("row count matches sample_submission", len(sub)==len(sub_template),
      f"({len(sub)} vs {len(sub_template)})")
check("columns identical", list(sub.columns)==list(sub_template.columns))
check("image_id order preserved",
      (sub[SUB_ID].astype(str).values==sub_template[SUB_ID].astype(str).values).all())
check("no NaNs", sub.isna().sum().sum()==0)
check("lat within [-90,90]", sub[SUB_LAT].between(-90,90).all())
check("lon within [-180,180]", sub[SUB_LON].between(-180,180).all())
check("radius > 0", (sub[SUB_RAD]>0).all())

r = sub[SUB_RAD].values
n_cap = int((r >= R_MAX-1).sum()); n_uniq = len(np.unique(np.round(r,1)))
check("radii NOT all pinned at cap", n_cap < 0.5*len(r), f"({n_cap}/{len(r)} at {R_MAX:.0f})")
check("radii varied", n_uniq > 0.3*len(r), f"({n_uniq} distinct)")
print(f"  radius  min {r.min():7.0f} | median {np.median(r):7.0f} | max {r.max():7.0f} km")

lat_u = len(np.unique(np.round(sub[SUB_LAT].values,3)))
check("coordinates varied", lat_u > 0.3*len(sub), f"({lat_u} distinct latitudes)")

n_ocean_left = 0
try:
    if geoms:
        pts2 = shapely.points(sub[SUB_LON].values.astype(float), sub[SUB_LAT].values.astype(float))
        ins = np.zeros(len(pts2),bool); pr2 = tree.query(pts2,predicate="intersects"); ins[pr2[0]]=True
        n_ocean_left = int((~ins).sum())
        check("few points left in open ocean", n_ocean_left < 0.10*len(sub),
              f"({n_ocean_left}/{len(sub)})")
except Exception: pass

print("="*72)
print("ALL CHECKS PASSED - safe to submit" if ok_all
      else "SOMETHING FAILED - do NOT submit until you understand why")
print("="*72)
print(f"\nartifacts used : {ART}")
print(f"heads          : {len(HEADS)}")
print(f"backbones      : {'CLIP + DINOv2' if USE_DINO else 'CLIP only'}")
print(f"lambda / alpha : {LAM} / {ALPHA}")
print(f"submission     : {OUT}")

  [PASS] row count matches sample_submission (2448 vs 2448)
  [PASS] columns identical 
  [PASS] image_id order preserved 
  [PASS] no NaNs 
  [PASS] lat within [-90,90] 
  [PASS] lon within [-180,180] 
  [PASS] radius > 0 
  [PASS] radii NOT all pinned at cap (0/2448 at 3000)
  [PASS] radii varied (1927 distinct)
  radius  min     241 | median     583 | max    2014 km
  [PASS] coordinates varied (2288 distinct latitudes)
  [PASS] few points left in open ocean (175/2448)
ALL CHECKS PASSED - safe to submit

artifacts used : /kaggle/input/notebooks/mitraasrinivasan1367/geogs-v7/artifacts
heads          : 5
backbones      : CLIP + DINOv2
lambda / alpha : 0.0 / 4.200000000000001
submission     : /kaggle/working/submission.csv
